# Drishti — DR grader training on Kaggle

Trains the ordinal (CORAL) grader on real corpora and exports `grader.onnx`
for the API to serve.

## Before running

1. **Add the data.** Right panel → *Add Input* → search and attach:
   - `aptos2019-blindness-detection` (competition data)
   - `diabetic-retinopathy-detection` (EyePACS — large; optional for a first run)
   - IDRiD and Messidor-2 need manual upload as private datasets, since neither
     is redistributable on Kaggle.
2. **Turn on the GPU.** Settings → Accelerator → *GPU P100* (or T4 x2).
3. **Turn on internet** if you want pretrained ImageNet weights, which you do —
   the CNN will not converge from scratch on a cohort this size.

Nothing is downloaded: Kaggle mounts the corpora read-only under
`/kaggle/input`, and `dr.datasets` resolves them from there automatically.

In [ ]:
# Bring in the project. Either attach the repo as a Kaggle dataset and point
# PROJECT at it, or clone it if internet is enabled.
import os, sys, subprocess
from pathlib import Path

PROJECT = Path("/kaggle/working/drishti")
REPO_URL = ""   # e.g. "https://github.com/<you>/drishti.git"

if not PROJECT.exists():
    attached = [p for p in Path("/kaggle/input").glob("*/server/app.py")]
    if attached:
        import shutil
        shutil.copytree(attached[0].parent.parent, PROJECT)
    elif REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
    else:
        raise SystemExit("Attach the repo as a dataset or set REPO_URL above.")

sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print("project:", PROJECT)

In [ ]:
!pip install -q timm onnx 2>&1 | tail -2
import torch, timm
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## What the data looks like

Check the corpora resolve and inspect the grade imbalance before training —
these datasets are roughly 73% grade 0 and under 3% grade 4, which is the
single biggest obstacle to a model that detects sight-threatening disease.

In [ ]:
from dr import datasets as D

records = D.load(["aptos"])          # add "idrid", "eyepacs" as available
print(D.describe(records))

In [ ]:
from dr import splits as S

train_idx, val_idx = S.train_val_split(records, 0.2, seed=0)
# Fails loudly rather than reporting an inflated score: EyePACS has both eyes
# per patient and they are highly correlated.
S.assert_no_patient_leakage(records, train_idx, val_idx)
print(S.summarise_split(records, train_idx, val_idx))

## Cache the resized images (do this once)

Full-resolution JPEG decoding, not the GPU, is what makes an epoch slow. On
EyePACS this step is the difference between hours and minutes per epoch.
Skip it for a small APTOS-only run.

In [ ]:
# from dr.torchdata import build_cache
# build_cache(records, size=512, cache_dir="/kaggle/working/cache", workers=4)

## Train

`--pretrained` is not optional in practice. From random init the network
collapses to predicting a single class for every image — 20% accuracy, and
actively worse than having no CNN, since fusion would drag every grade toward
that class while Grad-CAM produced convincing saliency that meant nothing.
`train.py` refuses to save such a model.

Kaggle kills sessions at 12h. Every epoch checkpoints to `artifacts/last.pt`;
re-run the same cell with `--resume artifacts/last.pt` to continue.

In [ ]:
from dr.train import main as train_main

train_main([
    "--datasets", "aptos",
    "--size", "512",
    "--backbone", "tf_efficientnet_b3_ns",
    "--epochs", "12",
    "--batch-size", "12",
    "--lr", "3e-4",
    "--workers", "2",
    "--out", "/kaggle/working/artifacts",
    # "--cache-dir", "/kaggle/working/cache",
    # "--external", "messidor2",     # never trained on; true external score
    # "--resume", "/kaggle/working/artifacts/last.pt",
])

## Results

`metrics.json` holds the best epoch's validation scores. Read **QWK** first —
it is what this task is scored on and the only common metric that understands
the grades are ordinal — then referable sensitivity, which is what a screening
programme is accountable for.

In [ ]:
import json
from pathlib import Path

art = Path("/kaggle/working/artifacts")
metrics = json.loads((art / "metrics.json").read_text())
from dr import metrics as M
print(M.format_report(metrics["val"], "validation"))
print("
thresholds:", [round(t, 3) for t in metrics["thresholds"]])

In [ ]:
import json
history = json.loads((art / "history.json").read_text())
print(f"{'epoch':>5} {'loss':>8} {'QWK':>8} {'ref.sens':>9} {'ref.spec':>9}")
for h in history:
    print(f"{h['epoch']:>5} {h['train_loss']:>8.4f} {h['qwk']:>8.4f} "
          f"{h['referable_sensitivity']:>9.3f} {h['referable_specificity']:>9.3f}")

## Score the lesion segmenter

Only IDRiD can do this — it is the one public corpus with pixel-level lesion
masks. Grade-only corpora can assess the segmenter indirectly at best.

In [ ]:
# from dr.eval_lesions import main as eval_main
# eval_main([])

## Export for serving

`grader.onnx` plus `grader.json` (thresholds, backbone, input size) is
everything the API needs. The server loads the frozen graph and never imports
the training code.

Download both from the notebook output, drop them in `artifacts/` beside the
deployment, and restart — `/api/health` will report the model as available.

In [ ]:
from pathlib import Path
for f in sorted(Path("/kaggle/working/artifacts").iterdir()):
    print(f"{f.name:24} {f.stat().st_size/1e6:8.1f} MB")

In [ ]:
# Sanity check: the exported graph must agree with the checkpoint that was
# validated, or the served model is not the model you measured.
import numpy as np, torch
from dr import model as MD, metrics as M

art = Path("/kaggle/working/artifacts")
ck = torch.load(art / "best.pt", map_location="cpu")
net = MD.build(ck["backbone"], pretrained=False)
net.load_state_dict(ck["model"]); net.eval()

x = np.random.randn(2, 3, ck["size"], ck["size"]).astype(np.float32)
with torch.no_grad():
    torch_grade = M.coral_expected_grade(net(torch.from_numpy(x)).numpy())
onnx_grade, _ = MD.OnnxGrader(art / "grader.onnx")(x)
print("max |torch - onnx| =", float(np.abs(torch_grade - onnx_grade).max()))
assert np.allclose(torch_grade, onnx_grade, atol=1e-4), "exported graph diverged"
print("parity OK")